# MotoGP Riders Finishing Positions - Dataset Exploration

This notebook explores the detailed finishing positions dataset showing riders' performance across all position categories (1st through 6th and beyond).

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading and Structure

In [ ]:
# Load data from CSV
data_path = Path("../data/raw/riders_finishing_positions.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
print(f"Columns: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:")
df.describe()

## 2. Top Performers Analysis

In [ ]:
# Calculate total races and podiums
df['Total_Races'] = df[['Victories', 'NumberofSecond', 'NumberofThird', 'Numberof4th', 'Numberof5th', 'Numberof6th']].sum(axis=1)
df['Podiums'] = df[['Victories', 'NumberofSecond', 'NumberofThird']].sum(axis=1)
df['Points_Finishes'] = df[['Victories', 'NumberofSecond', 'NumberofThird', 'Numberof4th', 'Numberof5th', 'Numberof6th']].sum(axis=1)
df['Podium_Rate'] = df['Podiums'] / df['Total_Races']
df['Win_Rate'] = df['Victories'] / df['Total_Races']

print("=== TOP RIDERS BY TOTAL RACES ===")
top_races = df.nlargest(15, 'Total_Races')[['Rider', 'Country', 'Total_Races', 'Victories', 'Podiums', 'Win_Rate', 'Podium_Rate']]
print(top_races)

In [ ]:
print("=== TOP RIDERS BY WIN RATE (min 20 races) ===")
df_min_races = df[df['Total_Races'] >= 20]
top_win_rate = df_min_races.nlargest(15, 'Win_Rate')[['Rider', 'Country', 'Total_Races', 'Victories', 'Win_Rate', 'Podium_Rate']]
print(top_win_rate)

print("\n=== TOP RIDERS BY PODIUM RATE (min 20 races) ===")
top_podium_rate = df_min_races.nlargest(15, 'Podium_Rate')[['Rider', 'Country', 'Total_Races', 'Podiums', 'Podium_Rate', 'Win_Rate']]
print(top_podium_rate)

## 3. Performance Categories Analysis

In [ ]:
# Riders who never won but had podiums
print("=== RIDERS WITH MOST PODIUMS BUT NO VICTORIES ===")
no_wins_podiums = df[(df['Victories'] == 0) & (df['Podiums'] > 0)]
no_wins_podiums = no_wins_podiums.nlargest(10, 'Podiums')[['Rider', 'Country', 'NumberofSecond', 'NumberofThird', 'Podiums']]
print(no_wins_podiums)

# Riders who never podiumed but had good finishes
print("\n=== RIDERS WITH MOST 4TH-6TH FINISHES BUT NO PODIUMS ===")
df['Non_Podium_Points'] = df[['Numberof4th', 'Numberof5th', 'Numberof6th']].sum(axis=1)
no_podium_points = df[(df['Podiums'] == 0) & (df['Non_Podium_Points'] > 0)]
no_podium_points = no_podium_points.nlargest(10, 'Non_Podium_Points')[['Rider', 'Country', 'Numberof4th', 'Numberof5th', 'Numberof6th', 'Non_Podium_Points']]
print(no_podium_points)

## 4. Country Analysis

In [ ]:
# Country performance analysis
print("=== COUNTRY PERFORMANCE ANALYSIS ===")
country_stats = df.groupby('Country').agg({
    'Victories': 'sum',
    'Podiums': 'sum',
    'Total_Races': 'sum',
    'Rider': 'count'
}).rename(columns={'Rider': 'Number_of_Riders'})

country_stats['Avg_Win_Rate'] = country_stats['Victories'] / country_stats['Total_Races']
country_stats['Avg_Podium_Rate'] = country_stats['Podiums'] / country_stats['Total_Races']
country_stats = country_stats.sort_values('Victories', ascending=False)

print("Top 15 countries by total victories:")
print(country_stats.head(15))

## 5. Visualizations

In [ ]:
# Position distribution for top riders
plt.figure(figsize=(15, 10))
top_10_riders = df.nlargest(10, 'Total_Races')
position_cols = ['Victories', 'NumberofSecond', 'NumberofThird', 'Numberof4th', 'Numberof5th', 'Numberof6th']

# Stacked bar chart
bottom = np.zeros(len(top_10_riders))
colors = ['gold', 'silver', '#CD7F32', '#4169E1', '#32CD32', '#FF6347']
labels = ['1st', '2nd', '3rd', '4th', '5th', '6th']

for i, col in enumerate(position_cols):
    plt.bar(top_10_riders['Rider'], top_10_riders[col], bottom=bottom, 
            color=colors[i], label=labels[i], alpha=0.8)
    bottom += top_10_riders[col]

plt.xlabel('Rider')
plt.ylabel('Number of Finishes')
plt.title('Finishing Position Distribution for Top 10 Riders (by total races)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Win rate vs Podium rate scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['Win_Rate'], df['Podium_Rate'], 
                     s=df['Total_Races']*2, alpha=0.6, c=df['Victories'], cmap='viridis')
plt.colorbar(scatter, label='Total Victories')

# Add labels for top performers
for idx, row in df.nlargest(8, 'Victories').iterrows():
    plt.annotate(row['Rider'].split()[-1], (row['Win_Rate'], row['Podium_Rate']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('Win Rate')
plt.ylabel('Podium Rate')
plt.title('Win Rate vs Podium Rate (bubble size = total races)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Country victories bar chart
plt.figure(figsize=(14, 8))
top_countries = country_stats.head(15)
plt.bar(top_countries.index, top_countries['Victories'], color='lightblue')
plt.xlabel('Country')
plt.ylabel('Total Victories')
plt.title('Total Victories by Country (Top 15)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Key Insights Summary

Based on the exploration of the finishing positions dataset:

### Dataset Characteristics
- Detailed breakdown of finishing positions (1st through 6th) for top riders
- Country information enables national performance analysis
- Allows calculation of various efficiency metrics (win rates, podium rates)

### Performance Patterns
- Clear distinction between elite winners and consistent performers
- Some riders excel at podiums but lack victories
- Others consistently score points without reaching podiums
- Win rates and podium rates vary significantly among top riders

### National Performance
- Certain countries dominate victory counts
- National averages reveal different competitive strengths
- Some nations have many riders with few elite performers
- Others have fewer riders but higher success rates

### Rider Categories Identified
- **Elite Winners**: High victory and podium counts
- **Consistent Podium Finishers**: Many podiums, fewer wins
- **Points Scorers**: Regular 4th-6th finishes without podiums
- **Specialists**: Riders with specific performance patterns

This analysis provides foundation for answering business questions about rider performance categories, national success patterns, and identifying riders who achieved success in specific finishing positions.